# ДЗ 1. Подготовка — Online Retail

Курс «Прогнозирование временных рядов и обработка сигналов».
Выполняем **пункт 6 «Подготовка»**: окружение, загрузка, обзор данных, выбор товара.

[Источник: UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)
— D. Chen (2015), [DOI: 10.24432/C5BW33](https://doi.org/10.24432/C5BW33), CC BY 4.0.
Строка — позиция счёта британского интернет-магазина, а не дневное наблюдение.
Товар с занятия **85123A** исключаем по уточнению пользователя.

Запуск: выбрать ядро **Python (hw1-retail)** и выполнить **Restart Kernel and Run All Cells**.
Все функции и операции находятся в этом Notebook.

In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
import pandas as pd
from IPython.display import display, Markdown

URL = 'https://archive.ics.uci.edu/static/public/352/online+retail.zip'
EXCLUDED, STOCK_CODE = '85123A', '20725'
pd.set_option('display.max_columns', 10)
print(f'pandas {pd.__version__}')

pandas 3.1.1


## Загрузка данных

Сначала загружаем ZIP с UCI. Если загрузка или чтение не удались,
используем `online+retail.zip` из папки Notebook.

In [2]:
def read_retail(data):
    with ZipFile(BytesIO(data)) as archive:
        return pd.read_excel(archive.open('Online Retail.xlsx'), dtype={
            'InvoiceNo': 'string', 'StockCode': 'string', 'CustomerID': 'Int64'
        }, parse_dates=['InvoiceDate'])


def load_retail():
    try:
        with urlopen(URL, timeout=30) as response:
            return read_retail(response.read()), 'UCI'
    except Exception as error:
        print(f'Используем локальный архив. Ошибка UCI: {error}')
        return read_retail(Path('online+retail.zip').read_bytes()), 'online+retail.zip'


df, source = load_retail()
print(f'Источник: {source}; размер: {df.shape}')
print('Период:', df.InvoiceDate.min(), '—', df.InvoiceDate.max())
df.head()

Источник: UCI; размер: (541909, 8)
Период: 2010-12-01 08:26:00 — 2011-12-09 12:50:00


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## Структура данных

| Поле | Значение |
| --- | --- |
| `InvoiceNo` | Номер счёта; префикс C означает отмену |
| `StockCode` | Код товара, может содержать буквы; используем для отбора |
| `Description` | Название товара |
| `Quantity` | Количество единиц в позиции; будущий показатель продаж |
| `InvoiceDate` | Дата и время счёта; основа будущей агрегации |
| `UnitPrice` | Цена единицы в фунтах стерлингов |
| `CustomerID` | Код покупателя; может отсутствовать |
| `Country` | Страна покупателя |

Идентификаторы счёта и товара читаем как строки, код покупателя — `Int64`
с поддержкой пропусков, дату — как дату и время.

In [3]:
display(pd.DataFrame({
    'Тип': df.dtypes.astype(str),
    'Пропуски': df.isna().sum(),
    'Пропуски, %': df.isna().mean().mul(100).round(2),
    'Уникальные значения': df.nunique(),
}))

,Тип,Пропуски,"Пропуски, %",Уникальные значения
InvoiceNo,string,0,0.00,25900
StockCode,string,0,0.00,4070
Description,object,1454,0.27,4223
Quantity,int64,0,0.00,722
InvoiceDate,datetime64[us],0,0.00,23260
UnitPrice,float64,0,0.00,1630
CustomerID,Int64,135080,24.93,4372
Country,str,0,0.00,38


## Выбор товара

Для сравнения товаров рассматриваем положительные продажи (`Quantity > 0`,
`UnitPrice > 0`, счёт без префикса C). Сравниваем число месяцев и дней с продажами.
Товар **85123A**, разобранный на занятии, исключаем. Исходную таблицу `df` сохраняем.

In [4]:
sales = df.loc[(df.Quantity > 0) & (df.UnitPrice > 0)
               & ~df.InvoiceNo.str.startswith('C', na=False)
               & df.StockCode.ne(EXCLUDED)].copy()
sales['day'] = sales.InvoiceDate.dt.normalize()
sales['month'] = sales.InvoiceDate.dt.to_period('M')
candidates = sales.groupby('StockCode').agg(
    name=('Description', 'first'), days=('day', 'nunique'),
    months=('month', 'nunique'), invoices=('InvoiceNo', 'nunique'),
    first=('InvoiceDate', 'min'), last=('InvoiceDate', 'max'),
).sort_values(['months', 'days'], ascending=False)
display(candidates.head(10))

,name,days,months,invoices,first,last
StockCode,,,,,,
22423,REGENCY CAKESTAND 3 TIER,301,13,1988,2010-12-01 12:27:00,2011-12-09 10:23:00
84879,ASSORTED COLOUR BIRD ORNAMENT,300,13,1455,2010-12-01 08:34:00,2011-12-09 12:31:00
85099B,JUMBO BAG RED RETROSPOT,300,13,2089,2010-12-01 09:57:00,2011-12-09 10:26:00
20725,LUNCH BAG RED RETROSPOT,298,13,1565,2010-12-01 09:37:00,2011-12-09 12:23:00
21212,PACK OF 72 RETROSPOT CAKE CASES,298,13,1320,2010-12-01 09:37:00,2011-12-09 10:23:00
22457,NATURAL SLATE HEART CHALKBOARD,297,13,1249,2010-12-01 09:53:00,2011-12-09 10:26:00
22960,JAM MAKING SET WITH JARS,295,13,1132,2010-12-01 08:34:00,2011-12-09 10:26:00
22197,SMALL POPCORN HOLDER,294,13,1392,2010-12-01 10:19:00,2011-12-09 12:19:00
22666,RECIPE BOX PANTRY YELLOW DESIGN,294,13,1111,2010-12-01 13:24:00,2011-12-09 10:26:00


In [5]:
product = df.loc[df.StockCode.eq(STOCK_CODE)].copy()
selected = candidates.loc[STOCK_CODE]
display(candidates.loc[[STOCK_CODE]])
print(f'Исходных строк товара, включая отмены: {len(product)}')
product.head()

,name,days,months,invoices,first,last
StockCode,,,,,,
20725,LUNCH BAG RED RETROSPOT,298,13,1565,2010-12-01 09:37:00,2011-12-09 12:23:00


Исходных строк товара, включая отмены: 1639


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
93,536378,20725,LUNCH BAG RED RETROSPOT,10,2010-12-01 09:37:00,1.65,14688,United Kingdom
368,536401,20725,LUNCH BAG RED RETROSPOT,1,2010-12-01 11:21:00,1.65,15862,United Kingdom
614,536412,20725,LUNCH BAG RED RETROSPOT,1,2010-12-01 11:49:00,1.65,17920,United Kingdom
1869,536544,20725,LUNCH BAG RED RETROSPOT,4,2010-12-01 14:32:00,4.21,<NA>,United Kingdom
2325,536577,20725,LUNCH BAG RED RETROSPOT,60,2010-12-01 16:13:00,1.45,13777,United Kingdom


In [6]:
display(Markdown(
    f'Выбран **{STOCK_CODE} — {selected["name"]}**. '
    f'Продажи есть в **{selected.months} месяцах** и **{selected.days} днях** '
    f'({selected["first"]:%d.%m.%Y}–{selected["last"]:%d.%m.%Y}). '
    'Длинная история и частые продажи подходят для дальнейшего сравнения частот. '
    'Они сами по себе не доказывают сезонность или пригодность для прогнозирования.'
))

Выбран **20725 — LUNCH BAG RED RETROSPOT**. Продажи есть в **13 месяцах** и **298 днях** (01.12.2010–09.12.2011). Длинная история и частые продажи подходят для дальнейшего сравнения частот. Они сами по себе не доказывают сезонность или пригодность для прогнозирования.

## Результат подготовки

Выбран **20725 — LUNCH BAG RED RETROSPOT**, отличный от **85123A** с занятия.
В `product` сохранены все транзакции выбранного товара, включая отмены.
Для будущего ряда используем количество единиц **Quantity**, все страны.

Дальше: определить правила учёта отмен и возвратов, построить дневную,
недельную и месячную агрегации. Отсутствие транзакций в день само по себе
не означает пропуск в данных.

Окружение и локальный Git-репозиторий подготовлены. Публикация на GitHub пока
не выполнена; инструкция в README.